In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from mlflow import MlflowClient
import numpy as np
import pandas as pd
import xgboost as xgb
import time
import joblib
import mlflow

### MLflow setup

In [3]:
client = MlflowClient(tracking_uri="http://127.0.0.1:8080/")    #connect to the tracking server
all_experiments = client.search_experiments()   #metadata associated with the Experiments on the server
experiments = [{
    "expr_name": expr.name, "life_cycle":expr.lifecycle_stage
} for expr in all_experiments]
print(experiments)

[{'expr_name': 'Resume text sentences classifier', 'life_cycle': 'active'}, {'expr_name': 'Default', 'life_cycle': 'active'}]


In [ ]:
# experiment Tags and metadata creation
experiment_description = (
    "This is the text sentence classifier project. "
    "This experiment contains the produce models for resume text classifier."
)
experiment_tags = {
    "project_name": "text-sentences-classifier",
    "project": "Resume Parser",
    "team": "3DSF",
    "project_Date": "9/16/2025",
    "mlflow.note.content": experiment_description,
}
text_sentences_classifier_expr = client.create_experiment(name="Augmented Dataset Text Sentence Classifier", tags=experiment_tags)

'226946552300919922'

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:8080")
augmented_dataset_expr = mlflow.set_experiment("Augmented Dataset Text Sentence Classifier")

In [6]:
#dataset processing
dataset = pd.read_csv('../Data/cvs_dataset/final_augmented_dataset.csv')
dataset = dataset.dropna()
dataset = dataset.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
features = [
    "token_count", "char_count", "avg_sentence_length", "avg_word_length",
    "punctuation_count", "comma_count", "period_count",
    "digit_count", "special_char_count", 
]
scaler = MinMaxScaler()
for f  in features:
    dataset[f] = scaler.fit_transform(dataset[[f]]).astype('float32')
dataset = dataset.drop('label', axis=1)
label_map = {label: idx for idx, label in enumerate(sorted(dataset['label_full'].unique()))}
print("labels: ", label_map)
dataset['target'] = dataset['label_full'].map(label_map)
dataset.head()

labels:  {'Education': 0, 'Experience': 1, 'Objective': 2, 'Personal Info': 3, 'Qualification & Certification': 4, 'Skills': 5, 'Summary': 6}


,text,token_count,char_count,avg_word_length,unique_word_ratio,stopword_ratio,punctuation_count,comma_count,period_count,digit_count,uppercase_ratio,bullet_point_flag,contains_year,special_char_count,avg_sentence_length,long_word_ratio,titlecase_ratio,label_full,target
0,Project under Graduation,0.013468,0.012968,0.084459,1.0000,0.2500,0.000000,0.000000,0.00,0.0,0.1200,0,0,0.000000,0.023529,0.5000,0.7500,Experience,1
1,Father’s Name,0.013468,0.007205,0.043919,1.0000,0.2500,0.000000,0.000000,0.00,0.0,0.3077,0,0,0.000000,0.023529,0.0000,0.5000,Personal Info,3
2,"Omgeo Connect, a complementary offering to Omg...",0.383838,0.411623,0.086486,0.6579,0.2544,0.096970,0.081967,0.12,0.0,0.0205,0,0,0.036364,0.167647,0.5175,0.1053,Experience,1
3,Implemented Singleton pattern for property loa...,0.060606,0.055716,0.072838,0.9444,0.2222,0.018182,0.032787,0.04,0.0,0.0515,0,0,0.000000,0.105882,0.4444,0.1667,Experience,1
4,· Developed report for Plan Vs Actual Billing ...,0.111111,0.093660,0.065541,0.6364,0.2727,0.012121,0.000000,0.08,0.0,0.0938,0,0,0.000000,0.097059,0.2121,0.2424,Experience,1


In [8]:
text_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), dtype=np.float32)
X_text = text_vectorizer.fit_transform(dataset['text'])
# trainig and test split
X = dataset.drop(['text', 'label_full', 'target'], axis=1)
full_X = np.hstack([X_text.toarray(), X.values])
y = dataset['target']
X_train, X_test, y_train, y_test = train_test_split(full_X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train shape: {y_train.shape}, Test shape: {y_test.shape}")

Train shape: (112714, 5016), Test shape: (48307, 5016)
Train shape: (112714,), Test shape: (48307,)


In [9]:
# Compute weights
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Convert to dict
class_weight_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weight_dict)

Class weights: {np.int64(0): np.float64(1.2597402597402598), np.int64(1): np.float64(0.6750230569296554), np.int64(2): np.float64(1.1377093195788879), np.int64(3): np.float64(1.037366318773354), np.int64(4): np.float64(1.5396825396825398), np.int64(5): np.float64(1.0626979936642027), np.int64(6): np.float64(0.7743952291636609)}


In [10]:
# decision tree model training
decision_tree_model = DecisionTreeClassifier(max_depth=30, min_samples_split=15, criterion='entropy', min_samples_leaf=5 ,random_state=42, class_weight=class_weight_dict)
t0 = time.time()
decision_tree_model.fit(X_train, y_train)
t_end = time.time()
y_pred = decision_tree_model.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
#mlflow tracking
model_params = {"max_depth": 30, "min_samples_split": 15, "min_samples_leaf": "entropy", "min_samples_leaf": 5}
model_metrics = {"Precision": precision, "Recall": recall, "F1": f1}
with mlflow.start_run(run_name="Decision_Tree") as run:
    mlflow.log_params(model_params)
    mlflow.log_metrics(model_metrics)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))
print(f"Precision: {precision}, Recall: {recall}, F1-score: {f1}")
print(f"Grid search took {t_end - t0:.2f} seconds")

🏃 View run Decision_Tree at: http://127.0.0.1:8080/#/experiments/226946552300919922/runs/bca4fec4434d4d7eb1083209dc97b6d6
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/226946552300919922
                               precision    recall  f1-score   support

                    Education       0.62      0.63      0.62      5478
                   Experience       0.65      0.61      0.63     10223
                    Objective       0.90      0.90      0.90      6066
                Personal Info       0.72      0.66      0.69      6652
Qualification & Certification       0.70      0.68      0.69      4483
                       Skills       0.46      0.64      0.54      6494
                      Summary       0.73      0.63      0.67      8911

                     accuracy                           0.67     48307
                    macro avg       0.68      0.68      0.68     48307
                 weighted avg       0.68      0.67      0.67     48307

Precision: 0.6819

##  Random Forst Classifier


In [11]:
rfc_model = RandomForestClassifier(random_state=42, n_estimators=200, max_depth=10, class_weight=class_weight_dict, criterion='gini')
rfc_model.fit(X_train, y_train)
y_pred = rfc_model.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
#mlflow tracking
model_params = {"max_depth": 30, "criterion": "gini", "n_estimators": 200}
model_metrics = {"Precision": precision, "Recall": recall, "F1": f1}
with mlflow.start_run(run_name="Random_forest") as run:
    mlflow.log_params(model_params)
    mlflow.log_metrics(model_metrics)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))
print(f"Precision: {precision}, Recall: {recall}, F1-score: {f1}")

🏃 View run Random_forest at: http://127.0.0.1:8080/#/experiments/226946552300919922/runs/9aef792e74524b239400901f9d263811
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/226946552300919922
                               precision    recall  f1-score   support

                    Education       0.61      0.63      0.62      5478
                   Experience       0.71      0.34      0.46     10223
                    Objective       0.85      0.87      0.86      6066
                Personal Info       0.42      0.83      0.56      6652
Qualification & Certification       0.61      0.56      0.59      4483
                       Skills       0.63      0.45      0.53      6494
                      Summary       0.59      0.64      0.61      8911

                     accuracy                           0.60     48307
                    macro avg       0.63      0.62      0.60     48307
                 weighted avg       0.63      0.60      0.59     48307

Precision: 0.6335

In [12]:
# Knn model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
#mlflow tracking
model_params = {"n_neighbors": 5}
model_metrics = {"Precision": precision, "Recall": recall, "F1": f1}
with mlflow.start_run(run_name="KNN") as run:
    mlflow.log_params(model_params)
    mlflow.log_metrics(model_metrics)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))
print(f"Precision: {precision}, Recall: {recall}, F1-score: {f1}")

🏃 View run KNN at: http://127.0.0.1:8080/#/experiments/226946552300919922/runs/e88d622823254bf28910310900491170
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/226946552300919922
                               precision    recall  f1-score   support

                    Education       0.69      0.83      0.75      5478
                   Experience       0.79      0.45      0.57     10223
                    Objective       0.96      0.97      0.96      6066
                Personal Info       0.55      0.86      0.67      6652
Qualification & Certification       0.85      0.87      0.86      4483
                       Skills       0.74      0.79      0.76      6494
                      Summary       0.90      0.79      0.84      8911

                     accuracy                           0.76     48307
                    macro avg       0.78      0.79      0.78     48307
                 weighted avg       0.79      0.76      0.76     48307

Precision: 0.78578750541036

In [13]:
xgb_classifier = xgb.XGBClassifier(tree_method="hist")
xgb_classifier.fit(X_train, y_train)
y_pred = xgb_classifier.predict(X_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
#mlflow tracking
model_params = {"tree_method": "hist"}
model_metrics = {"Precision": precision, "Recall": recall, "F1": f1}
with mlflow.start_run(run_name="XGB") as run:
    mlflow.log_params(model_params)
    mlflow.log_metrics(model_metrics)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))
print(f"Precision: {precision}, Recall: {recall}, F1-score: {f1}")

🏃 View run XGB at: http://127.0.0.1:8080/#/experiments/226946552300919922/runs/4029182fd25a40f2af2aa55902f6dc28
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/226946552300919922
                               precision    recall  f1-score   support

                    Education       0.81      0.77      0.79      5478
                   Experience       0.73      0.83      0.78     10223
                    Objective       0.94      0.94      0.94      6066
                Personal Info       0.80      0.81      0.81      6652
Qualification & Certification       0.87      0.76      0.81      4483
                       Skills       0.73      0.71      0.72      6494
                      Summary       0.79      0.74      0.76      8911

                     accuracy                           0.79     48307
                    macro avg       0.81      0.79      0.80     48307
                 weighted avg       0.80      0.79      0.80     48307

Precision: 0.79760021989266

In [11]:
# Saving the model
joblib.dump(text_vectorizer, '../saved_models/tfidf_vectorizer.pkl')
xgb_classifier.save_model('../saved_models/xgb_model.json')